# 04 - Model Training

## Load all dataset versions

In [6]:
# Imports

import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor


# Load processed data

x = pd.read_csv('../data/processed/x_train_processed.csv')
y = pd.read_csv('../data/processed/y_train.csv').squeeze()


# Models

configs = [
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression', Ridge(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42)),
    ('XGBoost', XGBRegressor(random_state=42)),
    ('Decision Tree',DecisionTreeRegressor(random_state= 42))
]


# Run comparison
results = {}

for name, model in configs:
    scores = cross_val_score(
        model,
        x,
        y,
        cv=10,
        scoring='neg_root_mean_squared_error'
    )

    results[name] = -scores.mean()


# Results

results_df = pd.Series(results, name='RMSE').sort_values()

print(results_df)

XGBoost              47691.884010
Random Forest        48958.603326
Decision Tree        67636.029149
Ridge Regression     70573.589845
Linear Regression    70575.246386
Name: RMSE, dtype: float64


In [2]:
from sklearn.metrics import root_mean_squared_error

In [3]:
final_model = XGBRegressor(random_state=42)

final_model.fit(x, y)

x_test = pd.read_csv('../data/processed/x_test_processed.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

test_preds = final_model.predict(x_test)

test_rmse = root_mean_squared_error(y_test, test_preds)

print("Final Test RMSE:", test_rmse)

Final Test RMSE: 48544.92945384295


In [7]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error

# Base model

xgb = XGBRegressor(random_state=42)


# Hyperparameter grid

param_grid = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
}

# Grid Search

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=10,
    n_jobs=-1,
    verbose=1
)


# Fit

grid_search.fit(x, y)


# Best parameters

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV RMSE:")
print(-grid_search.best_score_)

Fitting 10 folds for each of 243 candidates, totalling 2430 fits
Best Parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 700, 'subsample': 1.0}

Best CV RMSE:
45141.45749842036


In [11]:
final_model = XGBRegressor(
    n_estimators=700,
    max_depth=7,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=0.8,
    random_state=42
)

final_model.fit(x, y)

test_preds = final_model.predict(x_test)

test_rmse = root_mean_squared_error(y_test, test_preds)

print("Final Tuned XGBoost Test RMSE:", test_rmse)

Final Tuned XGBoost Test RMSE: 46669.83372282481
